# US Accidents · тяжесть ДТП, многоклассовый дисбаланс (Spark, 7.7 млн аварий)

**Данные:** [`sobhanmoosavi/us-accidents`](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) — ДТП на территории США (2016–2023),
собранные из потоков данных дорожных служб и камер: ~7.7 млн записей, ~3 ГБ CSV, 46 столбцов. Реальные события, не синтетика.

**Задача:** по месту, времени, погоде и дорожным объектам предсказать `Severity` — влияние аварии на трафик:
1 (минимальное) … 4 (максимальное, длительные заторы). Классов 4, и они **сильно несбалансированы**: подавляющее большинство — класс 2.

**Тема карточки — macro vs weighted.** Weighted-F1 и accuracy взвешены поддержкой классов и «не замечают» редкие;
macro-F1 усредняет классы поровну и штрафует за провал на редком. Здесь это видно на реальных числах.

**Сравниваем на Spark ML:** `LogReg` · `RandomForest` · `RandomForest` с весами классов · `RandomForest` + поправка на априорные вероятности (подбор на валидации).
Baseline — «всегда самый частый класс».

**Намеренно исключено (утечка цели):** `Distance(mi)` (протяжённость затора — следствие тяжести), `End_Time` (длительность), `Description`,
`Source` (разные провайдеры по-разному размечают тяжесть), `Year` (меняется состав провайдеров по годам — оставлен только для EDA).

In [ ]:
import os, re, sys, json, math, time, socket, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / 'requirements.txt').exists() and (parent / 'notebooks').exists():
            return parent
    raise RuntimeError('Корень проекта не найден')


SEED    = 42
PROJECT = find_project_root()
TASK    = 'accident_severity'

MODEL_DIR = PROJECT / 'models' / 'classification' / TASK
ART_DIR   = PROJECT / 'artifacts' / 'classification' / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT  :', PROJECT)
print('MODEL_DIR:', MODEL_DIR)
print('ART_DIR  :', ART_DIR)

## 1. Метрики без Spark (numpy)

Confusion matrix 4×4 считается на Spark одним `groupBy`; все остальные метрики (macro/weighted, MCC, κ, квадратичный κ, per-class) —
на драйвере из этих 16 чисел. Функции сверены с `sklearn` (расхождение ~1e-16).

In [ ]:
# ---------- метрики без Spark: считаются по гистограммам / confusion matrix ----------
import math, json
import numpy as np

NBINS = 1000          # разрешение порога = 0.001


def _div(a, b):
    return a / b if b else 0.0


def _fbeta(p, r, beta):
    d = beta * beta * p + r
    return (1 + beta * beta) * p * r / d if d else 0.0


def binary_metrics(tp, fp, fn, tn):
    """Все метрики бинарной классификации из четырёх чисел confusion matrix."""
    tp, fp, fn, tn = (float(x) for x in (tp, fp, fn, tn))
    n, P, N = tp + fp + fn + tn, tp + fn, tn + fp
    prec_p, rec_p = _div(tp, tp + fp), _div(tp, P)
    prec_n, rec_n = _div(tn, tn + fn), _div(tn, N)
    f1p, f05p, f2p = (_fbeta(prec_p, rec_p, b) for b in (1.0, 0.5, 2.0))
    f1n, f05n, f2n = (_fbeta(prec_n, rec_n, b) for b in (1.0, 0.5, 2.0))
    acc = _div(tp + tn, n)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = _div(tp * tn - fp * fn, den)
    pe = _div((tp + fp) * P + (tn + fn) * N, n * n)
    kappa = _div(acc - pe, 1 - pe)
    return {
        'accuracy': acc,
        'balanced_accuracy': (rec_p + rec_n) / 2,
        'precision_macro': (prec_p + prec_n) / 2,
        'recall_macro': (rec_p + rec_n) / 2,
        'f1_macro': (f1p + f1n) / 2,
        'f1_weighted': _div(f1p * P + f1n * N, n),
        'f0.5_macro': (f05p + f05n) / 2,
        'f2_macro': (f2p + f2n) / 2,
        'mcc': mcc,
        'kappa': kappa,
        'specificity': rec_n,
        'sensitivity': rec_p,
        'precision_pos': prec_p,
        'recall_pos': rec_p,
        'f1_pos': f1p,
        'f0.5_pos': f05p,
        'f2_pos': f2p,
    }


def hist_counts(neg, pos):
    """neg/pos — гистограммы длины NBINS+1 (бин = floor(score*NBINS)).
    Для порога k/NBINS (k = 0..NBINS) объект предсказан положительным при score >= k/NBINS.
    Возвращает массивы tp, fp, fn, tn длины NBINS+1."""
    neg = np.asarray(neg, dtype=np.float64)
    pos = np.asarray(pos, dtype=np.float64)
    neg_ge = np.cumsum(neg[::-1])[::-1]
    pos_ge = np.cumsum(pos[::-1])[::-1]
    P, N = pos.sum(), neg.sum()
    return pos_ge, neg_ge, P - pos_ge, N - neg_ge          # tp, fp, fn, tn


def _mcc_vec(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return np.divide(num, den, out=np.zeros_like(num, dtype=np.float64), where=den > 0)


def best_threshold_k(neg, pos, objective='mcc', cost_fp=1.0, cost_fn=1.0):
    """Индекс k лучшего порога на (валидационной) гистограмме.
    objective: 'mcc' (максимум) | 'cost' (минимум fp*cost_fp + fn*cost_fn)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    if objective == 'mcc':
        return int(np.argmax(_mcc_vec(tp, fp, fn, tn)))
    if objective == 'cost':
        return int(np.argmin(fp * cost_fp + fn * cost_fn))
    raise ValueError(objective)


def sweep_block(neg, pos, n_points=101):
    """Подвыборка порогов 0..1 для фронта: счётчики tp/fp/fn/tn на тестовой гистограмме."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    ks = np.unique(np.round(np.linspace(0, NBINS, n_points)).astype(int))
    return {
        'thresholds': [round(float(k) / NBINS, 4) for k in ks],
        'tp': [int(tp[k]) for k in ks], 'fp': [int(fp[k]) for k in ks],
        'fn': [int(fn[k]) for k in ks], 'tn': [int(tn[k]) for k in ks],
    }


def _thin(x, y):
    out_x, out_y = [], []
    for a, b in zip(x, y):
        if out_x and abs(a - out_x[-1]) < 2e-3 and abs(b - out_y[-1]) < 2e-3:
            continue
        out_x.append(round(float(a), 4)); out_y.append(round(float(b), 4))
    return out_x, out_y


def _cap(x, y, n=140):
    if len(x) <= n:
        return x, y
    idx = np.unique(np.linspace(0, len(x) - 1, n).round().astype(int))
    return [x[i] for i in idx], [y[i] for i in idx]


def curves_block(neg, pos):
    """ROC и PR-кривые по гистограмме (для графиков; точные AUC считает Spark)."""
    tp, fp, fn, tn = hist_counts(neg, pos)
    P, N = max(tp[0], 1.0), max(fp[0], 1.0)
    tpr, fpr = tp / P, fp / N
    prec = np.divide(tp, tp + fp, out=np.ones_like(tp, dtype=np.float64), where=(tp + fp) > 0)
    fx, fy = _cap(*_thin(fpr[::-1], tpr[::-1]))
    rx, ry = _cap(*_thin(tpr[::-1], prec[::-1]))
    return {'roc': {'fpr': fx, 'tpr': fy}, 'pr': {'recall': rx, 'precision': ry}}


def multiclass_metrics(cm):
    """cm[i][j] — сколько объектов истинного класса i предсказано как j."""
    cm = np.asarray(cm, dtype=np.float64)
    K, s = cm.shape[0], cm.sum()
    t, p, d = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    prec = np.divide(d, p, out=np.zeros(K), where=p > 0)
    rec = np.divide(d, t, out=np.zeros(K), where=t > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    acc = d.sum() / s
    mcc_den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0))
    mcc = (d.sum() * s - (p * t).sum()) / mcc_den if mcc_den else 0.0
    pe = (p * t).sum() / (s * s)
    kappa = (acc - pe) / (1 - pe) if pe < 1 else 0.0
    idx = np.arange(K)
    w = (idx[:, None] - idx[None, :]) ** 2 / max((K - 1) ** 2, 1)
    expected = np.outer(t, p) / s
    qwk = 1 - (w * cm).sum() / (w * expected).sum() if (w * expected).sum() else 0.0
    return {
        'accuracy': acc, 'balanced_accuracy': rec.mean(),
        'precision_macro': prec.mean(), 'recall_macro': rec.mean(), 'f1_macro': f1.mean(),
        'precision_weighted': (prec * t).sum() / s, 'recall_weighted': (rec * t).sum() / s,
        'f1_weighted': (f1 * t).sum() / s, 'mcc': mcc, 'kappa': kappa, 'qwk': qwk,
        '_per_class': {'precision': prec, 'recall': rec, 'f1': f1, 'support': t},
    }


def to_native(o):
    """Рекурсивно приводит numpy/NaN/Inf к JSON-совместимым значениям (NaN → null)."""
    if isinstance(o, dict):
        return {str(k): to_native(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_native(v) for v in o]
    if isinstance(o, np.ndarray):
        return to_native(o.tolist())
    if isinstance(o, np.generic):
        return to_native(o.item())
    if isinstance(o, float):
        return None if (math.isnan(o) or math.isinf(o)) else o
    if hasattr(o, '__fspath__'):
        return str(o)
    return o


def dump_json(obj, path):
    path.write_text(json.dumps(to_native(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

## 2. Spark: сессия, ресурсы, общий каталог

Executor: 8 ядер / 10 ГБ, драйвер 2 ГБ. 3 ГБ CSV читается распределённо; на драйвер возвращаются только таблицы в десятки строк и выборка для карты.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import vector_to_array
from pyspark import StorageLevel
import requests

# ---- бюджет ресурсов берём из docker-compose ----
# spark-worker:  command --cores 8 --memory 10g  (limit cpus=8, memory=12g)
# ml (driver):   лимит cpus=3.0, memory=6g       → драйвер 2g, локально не считаем
SPARK_MASTER_URL = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
CORES      = int(os.environ.get('SPARK_CORES', os.environ.get('WEATHER_SPARK_CORES', '8')))                 # worker: --cores 8
EXEC_MEM   = os.environ.get('SPARK_EXECUTOR_MEMORY', os.environ.get('WEATHER_SPARK_EXECUTOR_MEMORY', '8g'))  # worker: --memory 10g, limit 12g
DRIVER_MEM = os.environ.get('SPARK_DRIVER_MEMORY', '2g')       # контейнер ml ограничен 6g — драйвер держим скромным
LOCAL_CORES = int(os.environ.get('SPARK_LOCAL_CORES', '3'))    # контейнер ml: cpus 3.0 (только для SPARK_MODE=local)
FORCE_MODE = os.environ.get('SPARK_MODE', 'cluster')           # cluster (по умолчанию) | auto | local
ALLOW_LOCAL_FALLBACK = os.environ.get('ALLOW_LOCAL_FALLBACK', '0') == '1'
EVENT_DIR  = Path('/opt/spark/events')                         # общий каталог для Spark History Server


def _master_tcp_ok(url, timeout=2.0):
    try:
        host, port = url.split('://', 1)[1].split(':')
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except Exception:
        return False


def _alive_workers(url):
    """Список живых worker из Web UI мастера (8085). None — UI недоступен."""
    try:
        host = url.split('://', 1)[1].split(':')[0]
        info = requests.get(f'http://{host}:8085/json/', timeout=3).json()
        return [w for w in info.get('workers', []) if w.get('state') == 'ALIVE']
    except Exception:
        return None


def choose_mode():
    """
    cluster (default) — требуем живой spark-master с ALIVE-worker. Нет → RuntimeError.
    auto             — пробуем cluster, при неудаче fallback на local (с громким предупреждением).
    local            — принудительно local[3] (для отладки; урезает данные, ест 3 ядра ml).
    ALLOW_LOCAL_FALLBACK=1 дополнительно разрешает fallback в режиме cluster.
    """
    if FORCE_MODE == 'local':
        print(f'⚠ SPARK_MODE=local: работаем на {LOCAL_CORES} ядрах ml (данные урежутся, только для отладки)')
        return 'local'

    if FORCE_MODE not in ('cluster', 'auto'):
        raise ValueError(f'SPARK_MODE={FORCE_MODE!r} (ожидалось auto|cluster|local)')

    if _master_tcp_ok(SPARK_MASTER_URL):
        workers = _alive_workers(SPARK_MASTER_URL)
        if workers is None or len(workers) > 0:
            return 'cluster'
        print('⚠ spark-master отвечает, но ALIVE-worker нет — задачи выполнять некому')

    if FORCE_MODE == 'auto' or ALLOW_LOCAL_FALLBACK:
        print(f'⚠ падаю в fallback local[{LOCAL_CORES}] — это НЕ полный прогон, 3 ядра ml, DATA_FRACTION=0.15')
        return 'local'

    raise RuntimeError(
        'Spark-кластер недоступен, а SPARK_MODE=cluster (по умолчанию).\n'
        f'  master: {SPARK_MASTER_URL}\n'
        '  Запустите профиль:\n'
        '      docker compose --profile spark up -d\n'
        '  Проверить: http://localhost:8085 (master UI), http://localhost:8086 (worker UI)\n'
        '  Для локальной отладки без кластера:\n'
        '      SPARK_MODE=local     (урежет данные до 15%, 3 ядра ml)\n'
        '      SPARK_MODE=auto      (попробует cluster, потом local)\n'
    )


def start_spark(mode):
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    b = (SparkSession.builder.appName(f'ml-playground · {TASK}')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.driver.maxResultSize', '512m'))
    if mode == 'cluster':
        b = (b.master(SPARK_MASTER_URL)
              .config('spark.submit.deployMode', 'client')     # driver — здесь, executor — на worker
              .config('spark.driver.memory', DRIVER_MEM)
              .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))
              .config('spark.driver.bindAddress', '0.0.0.0')
              .config('spark.driver.port', '7078')
              .config('spark.blockManager.port', '7079')
              .config('spark.executor.memory', EXEC_MEM)       # 8g
              .config('spark.executor.cores', str(CORES))      # 8
              .config('spark.cores.max', str(CORES))           # 8 → 1 executor × 8 cores на worker
              .config('spark.sql.shuffle.partitions', str(CORES * 6))   # 48
              .config('spark.default.parallelism', str(CORES * 6)))     # 48
    else:
        b = (b.master(f'local[{LOCAL_CORES}]')
              .config('spark.driver.memory', '4g')
              .config('spark.sql.shuffle.partitions', str(LOCAL_CORES * 4)))
    if EVENT_DIR.is_dir():
        b = b.config('spark.eventLog.enabled', 'true').config('spark.eventLog.dir', f'file://{EVENT_DIR}')
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark


# ---------- ОБЩИЙ КАТАЛОГ (виден драйверу И executor-ам) ----------
# По docker-compose.yml в оба контейнера (ml и spark-worker) смонтированы:
#   ./data               → /opt/spark/data      (ml + worker)
#   ./models             → /app/models          (ml + worker)
#   ./spark-state/events → /opt/spark/events    (ml + worker)
# Плюс в ml есть `.:/app`, поэтому /app/data в ml — это ТОТ ЖЕ хост-каталог ./data,
# что и /opt/spark/data в ml и worker. Именно поэтому копировать ./data/flight-delays
# в ./data/spark_cache/raw/ незачем: worker уже видит его по /opt/spark/data/flight-delays.
#
# Имя проверочного файла — probe.txt, БЕЗ префикса '_'.
# Hadoop FileInputFormat молча игнорирует файлы с '_' или '.' в начале имени
# (то же правило прячет _SUCCESS и .crc в parquet), поэтому
#   spark.read.text('file:///.../_probe.txt').collect() == []
# и раньше это давало ложное «executor не видят каталог».
SHARED_CANDIDATES = [
    Path('/opt/spark/data/spark_cache'),       # ./data               → ml + worker
    Path('/app/models/.spark_cache'),          # ./models             → ml + worker
    Path('/opt/spark/events/spark_cache'),     # ./spark-state/events → ml + worker
    PROJECT / 'data' / 'spark_cache',          # fallback для local[3] / нестандартного compose
]


def pick_shared_dir(spark, mode):
    if mode == 'local':
        d = PROJECT / 'data' / 'spark_cache'
        d.mkdir(parents=True, exist_ok=True)
        print(f'✓ shared dir: {d} — local[3], executor в том же процессе')
        return d
    for d in SHARED_CANDIDATES:
        try:
            d.mkdir(parents=True, exist_ok=True)
            probe = d / 'probe.txt'                # НЕ _probe.txt (см. комментарий выше)
            probe.write_text('ok')
            got = spark.read.text(f'file://{probe}').collect()   # читает executor, не драйвер
            if got and got[0][0] == 'ok':
                print(f'✓ shared dir: {d} — виден executor-ам')
                return d
            print(f'  {d}: probe прочитан пустым (got={got!r})')
        except Exception as e:
            print(f'  {d}: executor не видят каталог ({type(e).__name__})')
    return None


MODE  = choose_mode()
spark = start_spark(MODE)
SHARED = pick_shared_dir(spark, MODE)

# fallback в local разрешён только через choose_mode (по SPARK_MODE/ALLOW_LOCAL_FALLBACK)
# если сюда попали с MODE=cluster и SHARED=None — это проблема монтирования, а не режима
if SHARED is None:
    raise RuntimeError(
        'Ни один общий каталог не виден executor-ам. Проверьте docker-compose:\n'
        '  spark-worker должен монтировать ./data, ./models или ./spark-state/events\n'
        '  по тому же пути, что и ml (см. SHARED_CANDIDATES).\n'
        '  Кандидаты: ' + ', '.join(str(p) for p in SHARED_CANDIDATES)
    )

sc = spark.sparkContext
DATA_FRACTION = float(os.environ.get('DATA_FRACTION', '1.0' if MODE == 'cluster' else '0.15'))

SPARK_INFO = {
    'mode': MODE,
    'master': SPARK_MASTER_URL if MODE == 'cluster' else f'local[{LOCAL_CORES}]',
    'spark_version': spark.version,
    'app_id': sc.applicationId,
    'cores': CORES if MODE == 'cluster' else LOCAL_CORES,
    'executor_memory': EXEC_MEM if MODE == 'cluster' else '4g (driver)',
    'default_parallelism': int(sc.defaultParallelism),
    'shuffle_partitions': int(spark.conf.get('spark.sql.shuffle.partitions')),
    'shared_dir': str(SHARED),
    'data_fraction': DATA_FRACTION,
}
print(json.dumps(SPARK_INFO, indent=2, ensure_ascii=False))
if MODE == 'local':
    print(f'\n⚠ local-режим: данные урезаны до {DATA_FRACTION:.0%} (DATA_FRACTION), используются {LOCAL_CORES} ядра ml. '
          'Для полного объёма на 8 ядрах spark-worker: docker compose --profile spark up -d и SPARK_MODE=cluster.')


# ---------- утилиты ----------
TIMINGS = {}


class stage:
    """with stage('имя'): ...  — печатает и копит время этапов (идёт в report.json)."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        print(f'▶ {self.name} ...')
        return self

    def __exit__(self, *exc):
        TIMINGS[self.name] = round(time.time() - self.t, 1)
        print(f'✔ {self.name}: {TIMINGS[self.name]} с')


def ensure_shared_dir(src):
    """
    Приводит src к пути, видимому executor-ам (под SHARED / /opt/spark/data).
    Копирует только когда действительно надо.

    Ключевой нюанс: в ml есть ДВА монтирования на один и тот же хост-каталог ./data —
       .:/app               → /app/data       (= ./data на хосте)
       ./data:/opt/spark/data → /opt/spark/data (= ./data на хосте)
    Поэтому /app/data/flight-delays и /opt/spark/data/flight-delays — ОДИН И ТОТ ЖЕ файл
    на хосте. spark-worker монтирует только ./data:/opt/spark/data, значит видит его как
    /opt/spark/data/flight-delays. Копирование в SHARED/raw/ тут не нужно и только плодит
    дубли; а попытка copytree на несуществующем пути — это та самая FileNotFoundError.
    """
    src = Path(src)
    try:
        src = src.resolve()
    except OSError:
        pass
    shared = SHARED.resolve()

    # 1) Уже под SHARED — отлично, ничего не делаем
    try:
        src.relative_to(shared)
        if src.exists():
            return src
    except ValueError:
        pass

    # 2) /app/data/... и /opt/spark/data/... — тот же хост-каталог ./data, что и у worker-а.
    #    Пересчитываем на /opt/spark/data/<rel> без копирования (работает и для файлов, и для каталогов).
    for alias in (Path('/app/data'), Path('/opt/spark/data')):
        try:
            rel = src.relative_to(alias.resolve())
        except ValueError:
            continue
        worker_path = Path('/opt/spark/data') / rel
        if worker_path.exists():
            print(f'✓ {src} → {worker_path} (тот же хост-каталог ./data, копия не нужна)')
            return worker_path

    # 3) Если src вообще не существует — внятная ошибка вместо падения copytree
    if not src.exists():
        raise FileNotFoundError(
            f'Нет данных: {src}\n'
            f'  kagglehub недоступен и локальной копии нет.\n'
            f'  Положите файлы на хост, например:\n'
            f'    ./data/{src.name}/    (в контейнере это /opt/spark/data/{src.name}/)\n'
            f'  или настройте доступ к Kaggle (KAGGLE_USERNAME / KAGGLE_KEY).\n'
        )

    # 4) Обычный кейс — копируем под SHARED/raw/<name>
    dst = shared / 'raw' / src.name
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists() and dst.resolve() == src:
            return src
        print(f'копирую файл {src} → {dst} (чтобы executor увидели файл)')
        shutil.copy2(src, dst)
        return dst
    if dst.exists() and dst.resolve() == src:
        return src
    print(f'копирую {src} → {dst} (чтобы executor увидели файлы)')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return dst


def spark_path(p):
    return f'file://{p}'

## 3. Загрузка и очистка

Читаем CSV одним проходом без `inferSchema` (иначе Spark прочитает 3 ГБ дважды), приводим типы вручную. Столбцы с круглыми скобками (`Temperature(F)`) переименовываем.
Время парсим по первым 19 символам — в части строк есть доли секунд.

Признаки:
* **гео/время:** широта, долгота, час, день недели, месяц, выходной, ночь;
* **погода:** температура, влажность, давление, видимость, ветер, осадки (пропуски — медианой по train);
* **дорожные объекты:** 12 булевых флагов (светофор, перекрёсток, пешеходный переход…);
* **штат** (индекс/one-hot) и **группа погодного условия** (топ-12 значений, остальное → `Other`).

Разбиение — случайное 70/10/20, фиксируется в parquet (`rand()` не пересчитывается).

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = str(SHARED / 'kagglehub')
import kagglehub

with stage('Загрузка us-accidents'):
    try:
        DATA_DIR = Path(kagglehub.dataset_download('sobhanmoosavi/us-accidents'))
    except Exception as e:
        print('kagglehub недоступен:', e)
        DATA_DIR = PROJECT / 'data' / 'us-accidents'
        print('использую локальную копию:', DATA_DIR)

DATA_DIR = ensure_shared_dir(DATA_DIR)
csv_files = sorted(DATA_DIR.rglob('*.csv'))
assert csv_files, f'CSV не найден в {DATA_DIR}'
CSV = max(csv_files, key=lambda p: p.stat().st_size)
print(f'{CSV.name}: {CSV.stat().st_size / 1e9:.2f} ГБ')

raw = spark.read.option('header', True).option('quote', '"').option('escape', '"').csv(spark_path(CSV))


def clean_name(c):
    return re.sub(r'[^0-9a-zA-Z]+', '_', c.strip()).strip('_').lower()


raw = raw.toDF(*[clean_name(c) for c in raw.columns])
NUM = ['temperature_f', 'humidity', 'pressure_in', 'visibility_mi', 'wind_speed_mph', 'precipitation_in']
POI = ['amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway', 'roundabout',
       'station', 'stop', 'traffic_calming', 'traffic_signal']
NEED = ['severity', 'start_time', 'start_lat', 'start_lng', 'state', 'weather_condition', 'sunrise_sunset'] + NUM + POI
missing = [c for c in NEED if c not in raw.columns]
assert not missing, f'в CSV нет столбцов {missing}; есть: {raw.columns}'
print('столбцов в CSV:', len(raw.columns))

In [ ]:
typed = (raw.select(
            F.col('severity').cast('int').alias('severity'),
            F.to_timestamp(F.substring(F.col('start_time'), 1, 19)).alias('ts'),
            F.col('start_lat').cast('double').alias('start_lat'),
            F.col('start_lng').cast('double').alias('start_lng'),
            F.col('state'), F.col('weather_condition'), F.col('sunrise_sunset'),
            *[F.col(c).cast('double').alias(c) for c in NUM],
            # POI: null в CSV (пустая ячейка) трактуем как отсутствие признака (0.0),
            # а не как null. Иначе VectorAssembler с handleInvalid='error' падал бы
            # на этих null — что и произошло в исходной версии.
            *[F.coalesce((F.col(c) == 'True').cast('double'), F.lit(0.0)).alias(c) for c in POI])
         .where(F.col('severity').isin(1, 2, 3, 4) & F.col('ts').isNotNull()
                & F.col('start_lat').isNotNull() & F.col('start_lng').isNotNull() & F.col('state').isNotNull()))

with stage('Подсчёт строк в CSV'):
    N_RAW = raw.count()
print(f'строк в CSV: {N_RAW:,}')

with stage('Топ погодных условий'):
    TOP_WEATHER = [r['weather_condition'] for r in typed.groupBy('weather_condition').count()
                   .where(F.col('weather_condition').isNotNull()).orderBy(F.desc('count')).limit(12).collect()]
print('топ-12 погодных условий:', TOP_WEATHER)

wc = F.col('weather_condition')
prepared = (typed
    .withColumn('label', (F.col('severity') - 1).cast('double'))
    .withColumn('weather_grp', F.when(wc.isNull(), 'Unknown').when(wc.isin(TOP_WEATHER), wc).otherwise('Other'))
    .withColumn('hour', F.hour('ts')).withColumn('dow', F.dayofweek('ts'))
    .withColumn('month', F.month('ts')).withColumn('year', F.year('ts'))
    .withColumn('is_weekend', F.col('dow').isin(1, 7).cast('double'))
    # is_night: null в sunrise_sunset трактуем как «день» (0.0). Без coalesce
    # null доходил до VectorAssembler и ронял всю сборку вектора.
    .withColumn('is_night', F.coalesce((F.col('sunrise_sunset') == 'Night').cast('double'), F.lit(0.0)))
    .withColumn('u', F.rand(SEED))
    .withColumn('fold', F.when(F.col('u') < 0.70, 'train').when(F.col('u') < 0.80, 'val').otherwise('test'))
    .drop('u', 'ts', 'weather_condition', 'sunrise_sunset'))
if DATA_FRACTION < 1.0:
    prepared = prepared.sample(False, DATA_FRACTION, SEED)

CLEAN_PATH = SHARED / 'accidents_clean.parquet'
with stage('Очистка, фолды, запись parquet'):
    prepared.write.mode('overwrite').parquet(spark_path(CLEAN_PATH))

df = spark.read.parquet(spark_path(CLEAN_PATH))
N_USED = df.count()
print(f'после очистки: {N_USED:,} ({N_USED / N_RAW:.1%} от CSV) · партиций: {df.rdd.getNumPartitions()}')

## 4. EDA на Spark: баланс классов и сдвиг по годам

Кроме распределения классов смотрим состав по годам: доли классов заметно меняются от года к году (меняется набор источников данных) —
поэтому `year` не используется как признак, а случайное разбиение оценивает «внутри распределения», а не «в будущее».

In [ ]:
CLASSES = ['S1', 'S2', 'S3', 'S4']
CLASS_DESC = {'S1': 'минимальное влияние на трафик', 'S2': 'небольшое', 'S3': 'заметное', 'S4': 'максимальное: длительные заторы'}
K = len(CLASSES)

with stage('EDA-агрегаты'):
    cls_rows = df.groupBy('label').count().collect()
    hour_rows = df.groupBy('hour').agg(F.count('*').alias('n'), F.avg((F.col('label') >= 2).cast('double')).alias('s34')).orderBy('hour').collect()
    state_rows = (df.groupBy('state').agg(F.count('*').alias('n'), F.avg((F.col('label') >= 2).cast('double')).alias('s34'))
                  .orderBy(F.desc('n')).limit(12).collect())
    year_rows = (df.groupBy('year').agg(F.count('*').alias('n'), *[F.avg((F.col('label') == float(k)).cast('double')).alias(f's{k + 1}') for k in range(K)])
                 .orderBy('year').collect())
    fold_rows = df.groupBy('fold').count().collect()

CLASS_COUNTS = {CLASSES[int(r['label'])]: int(r['count']) for r in cls_rows}
PRIORS = np.array([CLASS_COUNTS[c] for c in CLASSES], dtype=float) / N_USED
RARE = [c for c in CLASSES if CLASS_COUNTS[c] / N_USED < 0.05]
IMBALANCE = max(CLASS_COUNTS.values()) / max(min(CLASS_COUNTS.values()), 1)
print('распределение классов:')
for c in CLASSES:
    print(f'  {c} ({CLASS_DESC[c]:34s}) {CLASS_COUNTS[c]:>10,}  {CLASS_COUNTS[c] / N_USED:7.3%}{"  ← редкий" if c in RARE else ""}')
print(f'самый частый : самый редкий = {IMBALANCE:,.0f} : 1')

EDA = {
    'class_distribution': [{'class': c, 'n': CLASS_COUNTS[c], 'share': CLASS_COUNTS[c] / N_USED, 'desc': CLASS_DESC[c]} for c in CLASSES],
    'by_hour': [{'hour': int(r['hour']), 'n': int(r['n']), 's34': float(r['s34'])} for r in hour_rows],
    'by_state': [{'state': r['state'], 'n': int(r['n']), 's34': float(r['s34'])} for r in state_rows],
    'by_year': [{'year': int(r['year']), 'n': int(r['n']), **{f'S{k + 1}': float(r[f's{k + 1}']) for k in range(K)}} for r in year_rows],
}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].bar(CLASSES, [CLASS_COUNTS[c] for c in CLASSES], color=['#f43f5e' if c in RARE else '#6366f1' for c in CLASSES])
axes[0].set_yscale('log'); axes[0].set_title('Баланс классов (лог-шкала)')
for i, c in enumerate(CLASSES):
    axes[0].text(i, CLASS_COUNTS[c] * 1.1, f'{CLASS_COUNTS[c]:,}', ha='center', fontsize=8)
yd = pd.DataFrame(EDA['by_year']).set_index('year')[CLASSES]
yd.plot(kind='bar', stacked=True, ax=axes[1], color=['#f43f5e', '#6366f1', '#0ea5e9', '#f59e0b']); axes[1].set_title('Доли классов по годам (сдвиг состава)'); axes[1].legend(ncol=4, fontsize=8)
hd = pd.DataFrame(EDA['by_hour']); axes[2].plot(hd.hour, hd.s34 * 100, marker='o', color='#8b5cf6'); axes[2].set_title('Доля тяжёлых (S3+S4) по часу, %'); axes[2].set_xlabel('час')
plt.tight_layout(); plt.savefig(ART_DIR / 'eda.png', bbox_inches='tight'); plt.show()

## 5. Признаки

Трансформеры (медианы, индексы штатов и погоды) обучаются **только на train** и применяются ко всем выборкам.
Для деревьев — индексированные категории (`maxBins ≥ числа штатов`), для логистической регрессии — one-hot и циклические `sin/cos` часа.

**Веса классов:** `w_k = N / (K · n_k)` по train — редкому классу достаётся больший вес.

In [ ]:
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.stat import Summarizer

GEO_TIME = ['start_lat', 'start_lng', 'hour', 'dow', 'month', 'is_weekend', 'is_night']

with stage('Признаки: импутация, индексы, векторы, кэш'):
    train_df = df.where(F.col('fold') == 'train')
    imputer = Imputer(strategy='median', inputCols=NUM, outputCols=[f'{c}_imp' for c in NUM]).fit(train_df)
    s_idx = StringIndexer(inputCol='state', outputCol='state_idx', handleInvalid='keep').fit(train_df)
    w_idx = StringIndexer(inputCol='weather_grp', outputCol='weather_idx', handleInvalid='keep').fit(train_df)
    MAX_BINS = max(64, len(s_idx.labelsArray[0]) + 2)

    feat = w_idx.transform(s_idx.transform(imputer.transform(df)))
    feat = (OneHotEncoder(inputCols=['state_idx', 'weather_idx'], outputCols=['state_ohe', 'weather_ohe'], handleInvalid='keep')
            .fit(feat).transform(feat))
    two_pi = 2 * math.pi
    feat = (feat.withColumn('hour_sin', F.sin(F.col('hour') * two_pi / 24)).withColumn('hour_cos', F.cos(F.col('hour') * two_pi / 24)))

    IMP = [f'{c}_imp' for c in NUM]
    TREE_FEATURES = IMP + GEO_TIME + POI + ['state_idx', 'weather_idx']
    LR_NUMERIC = IMP + ['start_lat', 'start_lng', 'hour_sin', 'hour_cos', 'dow', 'month', 'is_weekend', 'is_night'] + POI
    TREE_NAMES = [n.replace('_imp', '') for n in TREE_FEATURES]

    feat = VectorAssembler(inputCols=TREE_FEATURES, outputCol='features').transform(feat)
    feat = VectorAssembler(inputCols=LR_NUMERIC + ['state_ohe', 'weather_ohe'], outputCol='features_lr').transform(feat)

    cnt_tr = {int(r['label']): int(r['count']) for r in feat.where(F.col('fold') == 'train').groupBy('label').count().collect()}
    N_TR_ALL = sum(cnt_tr.values())
    CLASS_W = {k: N_TR_ALL / (K * max(cnt_tr.get(k, 1), 1)) for k in range(K)}
    feat = feat.withColumn('w', F.coalesce(*[F.when(F.col('label') == float(k), F.lit(float(w))) for k, w in CLASS_W.items()]))

    EXTRA = ['severity', 'start_lat', 'start_lng', 'hour', 'state', 'temperature_f', 'visibility_mi']
    feat = feat.select('label', 'fold', 'features', 'features_lr', 'w', *EXTRA)
    train = feat.where(F.col('fold') == 'train').persist(StorageLevel.MEMORY_AND_DISK)
    val   = feat.where(F.col('fold') == 'val').persist(StorageLevel.MEMORY_AND_DISK)
    test  = feat.where(F.col('fold') == 'test').persist(StorageLevel.MEMORY_AND_DISK)
    N_TRAIN, N_VAL, N_TEST = train.count(), val.count(), test.count()

LR_NAMES = LR_NUMERIC + [f'state={s}' for s in s_idx.labelsArray[0]] + [f'weather={w}' for w in w_idx.labelsArray[0]]
LR_STD = np.array(train.select(Summarizer.metrics('std').summary(F.col('features_lr')).alias('s')).first()['s']['std'].toArray())
print(f'train={N_TRAIN:,}  val={N_VAL:,}  test={N_TEST:,}')
print('признаков (деревья):', len(TREE_FEATURES), '| (LogReg после one-hot):', len(LR_NAMES))
print('веса классов:', {CLASSES[k]: round(w, 2) for k, w in CLASS_W.items()})

## 6. Модели

* **LogReg** — мультиномиальная логистическая регрессия на всём train.
* **RandomForest** — без весов. Деревья учатся на выборке train не более `TREE_TRAIN_CAP` строк (ограничение памяти executor 8 ГБ).
* **RandomForest · веса** — тот же лес с `weightCol`.
* **RandomForest · prior-коррекция** — без переобучения: вероятности делятся на `prior_k^α`, α подбирается на валидации по macro-F1.
  Это «бесплатный» способ сдвинуть решение в пользу редких классов.
* **Baseline** — всегда самый частый класс.

Выбор лучшей модели — по **macro-F1** на тесте (hero), но в отчёт попадают все метрики, включая weighted-F1 и accuracy, чтобы расхождение было видно.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

TREE_TRAIN_CAP = int(os.environ.get('TREE_TRAIN_CAP', 2_500_000))
frac_tree = min(1.0, TREE_TRAIN_CAP / N_TRAIN)
tree_train = train.sample(False, frac_tree, SEED).persist(StorageLevel.MEMORY_AND_DISK) if frac_tree < 1.0 else train
print(f'деревья учатся на {frac_tree:.0%} train (≈{int(N_TRAIN * frac_tree):,} строк)')


def predict_df(model, sdf, alpha=None, extra=()):
    """label, prediction, p (вероятности классов), adj (скорректированные) + extra."""
    out = model.transform(sdf).select('label', *extra, vector_to_array('probability').alias('p'), F.col('prediction').alias('pred_raw'))
    if alpha is None:
        return out.withColumn('prediction', F.col('pred_raw')).withColumn('adj', F.col('p'))
    adj = F.array(*[F.col('p')[k] / F.lit(float(PRIORS[k] ** alpha)) for k in range(K)])
    return (out.withColumn('adj', adj)
               .withColumn('prediction', F.expr('cast(array_position(adj, array_max(adj)) - 1 as double)')))


def confusion(sdf):
    cm = np.zeros((K, K))
    for r in sdf.groupBy('label', 'prediction').count().collect():
        cm[int(r['label']), int(r['prediction'])] = r['count']
    return cm


def top2_accuracy(sdf):
    r = (sdf.withColumn('p_true', F.expr('element_at(adj, cast(label as int) + 1)'))
            .withColumn('rank', F.expr('size(filter(adj, x -> x > p_true))'))
            .agg(F.avg((F.col('rank') < 2).cast('double')).alias('t2')).first())
    return float(r['t2'])


def tree_importance(model, names):
    imp = model.featureImportances.toArray()
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(names, imp)], key=lambda d: -d['value'])


def lr_importance(model):
    coef = np.abs(model.coefficientMatrix.toArray()).mean(axis=0) * LR_STD
    coef = coef / coef.sum() if coef.sum() else coef
    return sorted([{'name': n, 'value': float(v)} for n, v in zip(LR_NAMES, coef)], key=lambda d: -d['value'])


def make_entry(name, family, cm, top2=None, importance=None, train_s=None, note=None, is_baseline=False, alpha=None):
    mm = multiclass_metrics(cm)
    pc = mm.pop('_per_class')
    if top2 is not None:
        mm['top2_accuracy'] = top2
    return {
        'name': name, 'model': family, 'note': note, 'is_baseline': bool(is_baseline), 'is_best': False, 'alpha': alpha,
        'metrics': mm,
        'per_class': {c: {'precision': float(pc['precision'][i]), 'recall': float(pc['recall'][i]),
                          'f1': float(pc['f1'][i]), 'support': int(pc['support'][i])} for i, c in enumerate(CLASSES)},
        'confusion_matrix': cm.astype(int).tolist(),
        'feature_importance': (importance or [])[:25],
        'train_seconds': train_s,
    }


rf_kw = dict(featuresCol='features', labelCol='label', numTrees=40, maxDepth=12, maxBins=MAX_BINS,
             subsamplingRate=0.25, minInstancesPerNode=20, featureSubsetStrategy='sqrt', seed=SEED)
entries, fitted, SCORERS = [], {}, {}


def train_and_eval(name, family, est, tr, note, importance_fn):
    with stage(f'обучение · {name}'):
        model = est.fit(tr)
    train_s = TIMINGS[f'обучение · {name}']
    with stage(f'оценка · {name}'):
        sc = predict_df(model, test).persist()
        cm, t2 = confusion(sc), top2_accuracy(sc)
        sc.unpersist()
    entries.append(make_entry(name, family, cm, t2, importance_fn(model), train_s, note))
    fitted[name], SCORERS[name] = model, (model, None)
    e = entries[-1]['metrics']
    print(f'   {name}: macro-F1={e["f1_macro"]:.4f}  weighted-F1={e["f1_weighted"]:.4f}  accuracy={e["accuracy"]:.4f}  MCC={e["mcc"]:.4f}')
    return model


lr_model = train_and_eval('LogReg', 'LogisticRegression',
                          LogisticRegression(featuresCol='features_lr', labelCol='label', family='multinomial', maxIter=50, regParam=1e-4),
                          train, 'мультиномиальная регрессия, без весов', lr_importance)
rf_model = train_and_eval('RandomForest', 'RandomForest', RandomForestClassifier(**rf_kw), tree_train,
                          'без весов', lambda m: tree_importance(m, TREE_NAMES))
train_and_eval('RandomForest · веса', 'RandomForest', RandomForestClassifier(weightCol='w', **rf_kw), tree_train,
               'weightCol: w_k = N / (K·n_k)', lambda m: tree_importance(m, TREE_NAMES))

# --- prior-коррекция: α подбирается на валидации по macro-F1 ---
with stage('prior-коррекция · подбор α на валидации'):
    val_p = predict_df(rf_model, val).persist()
    alpha_scores = {}
    for a in (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5):
        adj = F.array(*[F.col('p')[k] / F.lit(float(PRIORS[k] ** a)) for k in range(K)])
        v = val_p.withColumn('adj', adj).withColumn('prediction', F.expr('cast(array_position(adj, array_max(adj)) - 1 as double)'))
        alpha_scores[a] = multiclass_metrics(confusion(v))['f1_macro']
    val_p.unpersist()
ALPHA = max(alpha_scores, key=alpha_scores.get)
print('macro-F1 на валидации по α:', {a: round(s, 4) for a, s in alpha_scores.items()}, '→ α =', ALPHA)

with stage('оценка · RandomForest · prior-коррекция'):
    sc = predict_df(rf_model, test, alpha=ALPHA).persist()
    cm, t2 = confusion(sc), top2_accuracy(sc)
    sc.unpersist()
entries.append(make_entry(f'RandomForest · prior α={ALPHA}', 'RandomForest', cm, t2,
                          tree_importance(rf_model, TREE_NAMES), None, f'p_k / prior_k^α, α={ALPHA} подобран на валидации', alpha=ALPHA))
SCORERS[entries[-1]['name']] = (rf_model, ALPHA)
e = entries[-1]['metrics']
print(f'   {entries[-1]["name"]}: macro-F1={e["f1_macro"]:.4f}  weighted-F1={e["f1_weighted"]:.4f}  accuracy={e["accuracy"]:.4f}  MCC={e["mcc"]:.4f}')

# --- baseline: всегда самый частый класс ---
test_support = np.array(entries[0]['confusion_matrix']).sum(axis=1)
maj = int(np.argmax(test_support))
cm0 = np.zeros((K, K)); cm0[:, maj] = test_support
entries.append(make_entry(f'Baseline · всегда {CLASSES[maj]}', 'Baseline', cm0, float(np.sort(test_support)[-2:].sum() / test_support.sum()),
                          None, None, 'предсказывает самый частый класс', is_baseline=True))

trained = [e for e in entries if not e['is_baseline']]
best = max(trained, key=lambda e: e['metrics']['f1_macro'])
best['is_best'] = True
print('\nЛучшая по macro-F1:', best['name'], f'{best["metrics"]["f1_macro"]:.4f}')

## 7. Сводка

Таблица показывает главное: **accuracy и weighted-F1 у всех моделей (и у baseline) близки и высоки, macro-F1 — нет.**
Колонка `gap` = weighted-F1 − macro-F1: чем она больше, тем сильнее «взвешенные» метрики скрывают провал на редких классах.

In [ ]:
tbl = pd.DataFrame([{
    'model': e['name'], 'macro-F1': round(e['metrics']['f1_macro'], 4), 'weighted-F1': round(e['metrics']['f1_weighted'], 4),
    'gap': round(e['metrics']['f1_weighted'] - e['metrics']['f1_macro'], 4), 'accuracy': round(e['metrics']['accuracy'], 4),
    'balanced acc.': round(e['metrics']['balanced_accuracy'], 4), 'MCC': round(e['metrics']['mcc'], 4),
    'QWK': round(e['metrics']['qwk'], 4),
    **{f'recall {c}': round(e['per_class'][c]['recall'], 3) for c in CLASSES},
} for e in entries]).set_index('model')
display(tbl)
tbl.to_csv(ART_DIR / 'metrics_all_models.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(entries)); w = 0.38
axes[0].bar(x - w / 2, [e['metrics']['f1_macro'] for e in entries], w, label='macro-F1', color='#8b5cf6')
axes[0].bar(x + w / 2, [e['metrics']['f1_weighted'] for e in entries], w, label='weighted-F1', color='#34d399')
axes[0].set_xticks(x); axes[0].set_xticklabels([e['name'] for e in entries], rotation=20, ha='right', fontsize=8)
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].set_title('macro-F1 vs weighted-F1')
rec = pd.DataFrame({e['name']: [e['per_class'][c]['recall'] for c in CLASSES] for e in entries}, index=CLASSES)
rec.plot(kind='bar', ax=axes[1], width=.85); axes[1].set_ylim(0, 1); axes[1].set_title('Recall по классам'); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(ART_DIR / 'macro_vs_weighted.png', bbox_inches='tight'); plt.show()

cm = np.array(best['confusion_matrix']); cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, cbar=False)
ax.set_xlabel('предсказано'); ax.set_ylabel('истинный'); ax.set_title(f'{best["name"]} · recall по строкам')
plt.tight_layout(); plt.savefig(ART_DIR / 'confusion_matrix.png', bbox_inches='tight'); plt.show()

## 8. Выборка для карты и сохранение отчёта

Для визуализации во фронте берём **стратифицированную выборку** тестовых аварий (до ~250 на класс) с предсказанием лучшей модели:
координаты, час, штат, истинный и предсказанный класс. Это единственные строки данных, которые попадают на драйвер.

In [ ]:
model_b, alpha_b = SCORERS[best['name']]
test_counts = {k: int(np.array(best['confusion_matrix'])[k].sum()) for k in range(K)}
fractions = {float(k): min(1.0, 250 / max(n, 1)) for k, n in test_counts.items()}
with stage('Выборка для карты (toPandas)'):
    sp = (predict_df(model_b, test, alpha=alpha_b, extra=EXTRA)
          .sampleBy('label', fractions, SEED)
          .select('label', 'prediction', 'start_lat', 'start_lng', 'hour', 'state', 'temperature_f', 'visibility_mi', 'p')
          .limit(1500).toPandas())

SAMPLES = [{
    'lat': float(r.start_lat), 'lng': float(r.start_lng), 'hour': int(r.hour), 'state': r.state,
    'temperature_f': None if pd.isna(r.temperature_f) else float(r.temperature_f),
    'visibility_mi': None if pd.isna(r.visibility_mi) else float(r.visibility_mi),
    'true': CLASSES[int(r.label)], 'predicted': CLASSES[int(r.prediction)], 'correct': bool(r.label == r.prediction),
    'proba': {c: float(p) for c, p in zip(CLASSES, r.p)},
} for r in sp.itertuples()]
print('точек для карты:', len(SAMPLES), '· верно:', f'{np.mean([s["correct"] for s in SAMPLES]):.1%}')

In [ ]:
def pct_(x, d=1):
    return f'{x * 100:.{d}f}%'


naive = next(e for e in entries if e['is_baseline'])
bm, nm = best['metrics'], naive['metrics']
rare_info = {c: best['per_class'][c] for c in RARE}
gap_best = bm['f1_weighted'] - bm['f1_macro']
acc_spread = max(e['metrics']['accuracy'] for e in entries) - min(e['metrics']['accuracy'] for e in entries)
by_name = {e['name']: e['metrics'] for e in entries}
plain_rf = by_name['RandomForest']

SPARK_BLOCK = {**SPARK_INFO, 'n_rows_raw': int(N_RAW), 'n_rows_used': int(N_USED), 'timings': TIMINGS,
               'tree_train_rows': int(N_TRAIN * frac_tree), 'ui': {'master': ':8085', 'history': ':18080'}}

narrative = {
    'context': (f'US Accidents: {N_RAW:,} записей, после очистки {N_USED:,}. Классов 4 (Severity 1–4); распределение: ' +
                ', '.join(f'{c} — {pct_(CLASS_COUNTS[c] / N_USED, 2)}' for c in CLASSES) +
                f'. Редкие классы: {", ".join(RARE)}. Признаки — место, время, погода и дорожные объекты; '
                'протяжённость затора, длительность и источник данных исключены как утечка.'),
    'hero_metric': 'macro-F1',
    'why_hero': ('macro-F1 усредняет F1 по классам с равным весом, поэтому провал на редком классе сразу снижает итог. '
                 'Weighted-F1 и accuracy взвешены поддержкой и в основном отражают качество на самом частом классе.'),
    'why_not_weighted': (f'У «{best["name"]}» weighted-F1 = {bm["f1_weighted"]:.3f}, macro-F1 = {bm["f1_macro"]:.3f} (разрыв {gap_best:.3f}). '
                         f'Baseline «всегда {CLASSES[maj]}» получает accuracy {pct_(nm["accuracy"])} и weighted-F1 {nm["f1_weighted"]:.3f}, '
                         f'но macro-F1 всего {nm["f1_macro"]:.3f}. Разброс accuracy между всеми моделями — {acc_spread:.3f}.'),
    'prior_correction': (f'Поправка на априорные вероятности (α={ALPHA}) без переобучения меняет macro-F1 у леса с {plain_rf["f1_macro"]:.3f} до '
                         f'{by_name[f"RandomForest · prior α={ALPHA}"]["f1_macro"]:.3f}: решение смещается в сторону редких классов ценой точности на частом.'),
    'main_conclusion': (f'Лучшая по macro-F1 — {best["name"]}: macro-F1={bm["f1_macro"]:.4f}, weighted-F1={bm["f1_weighted"]:.4f}, '
                        f'accuracy={pct_(bm["accuracy"])}, MCC={bm["mcc"]:.3f}, квадратичный κ={bm["qwk"]:.3f}. ' +
                        '; '.join(f'recall {c}: {pct_(v["recall"])}' for c, v in rare_info.items()) + '.'),
    'rules': {
        'hero_macro_f1': 'Hero-метрика macro-F1: каждый класс весит одинаково',
        'weighted_hides': 'Weighted-F1 и accuracy показываем рядом — разрыв и есть диагноз',
        'no_leakage': 'Distance, End_Time, Source, Year исключены из признаков',
        'ordinal': 'Классы упорядочены: квадратичный κ штрафует ошибку на 3 ступени сильнее, чем на 1',
    },
}

DIAG = {
    'is_balanced': False, 'imbalance_ratio': float(IMBALANCE), 'rare_classes': RARE,
    'naive_accuracy': float(nm['accuracy']), 'naive_f1_macro': float(nm['f1_macro']), 'naive_f1_weighted': float(nm['f1_weighted']),
    'best_f1_macro': float(bm['f1_macro']), 'best_f1_weighted': float(bm['f1_weighted']),
    'macro_weighted_gap': float(gap_best), 'best_accuracy': float(bm['accuracy']),
    'accuracy_spread_models': float(acc_spread),
    'weighted_hides_rare': bool(gap_best > 0.05),
    'accuracy_misleads': bool(nm['accuracy'] > min(e['metrics']['accuracy'] for e in trained)),
    'rare_recall': {c: float(v['recall']) for c, v in rare_info.items()},
    'alpha': ALPHA,
}

report = {
    'meta': {
        'task': TASK,
        'title': 'US Accidents · тяжесть ДТП',
        'subtitle': 'Многоклассовый дисбаланс: macro-F1 видит редкие классы, weighted-F1 и accuracy — нет',
        'dataset': 'sobhanmoosavi/us-accidents (Kaggle · 2016–2023)',
        'target': 'Severity (1–4)',
        'classes': CLASSES, 'class_descriptions': CLASS_DESC, 'rare_classes': RARE,
        'features': TREE_NAMES, 'seed': SEED, 'generated_at': datetime.now().isoformat(),
        'n_samples_raw': int(N_RAW), 'n_samples': int(N_USED), 'n_features': len(TREE_NAMES), 'n_classes': K,
        'n_train': int(N_TRAIN), 'n_val': int(N_VAL), 'n_test': int(N_TEST),
        'target_stats': {'class_distribution': CLASS_COUNTS, 'imbalance_ratio': float(IMBALANCE),
                         'priors': {c: float(p) for c, p in zip(CLASSES, PRIORS)}, 'class_weights': {CLASSES[k]: float(w) for k, w in CLASS_W.items()}},
        'split_note': 'Случайное разбиение 70/10/20; fold зафиксирован в parquet. Оценка «внутри распределения», без сдвига по времени',
        'spark': SPARK_BLOCK,
        'narrative': narrative,
    },
    'headline': {
        'best_model': best['name'], 'best_metric': 'f1_macro', 'best_metric_value': float(bm['f1_macro']),
        'verdict': (f'Лучшая по macro-F1 — {best["name"]} · macro-F1={bm["f1_macro"]:.4f} · weighted-F1={bm["f1_weighted"]:.4f} · '
                    f'accuracy={bm["accuracy"]:.4f} · MCC={bm["mcc"]:.3f}'),
        'diagnostic_verdict': DIAG,
    },
    'models': sorted(entries, key=lambda e: (e['is_baseline'], -e['metrics']['f1_macro'])),
    'eda': EDA,
    'sample_predictions': SAMPLES,
    'charts': {k.replace('.png', ''): k for k in ('eda.png', 'macro_vs_weighted.png', 'confusion_matrix.png') if (ART_DIR / k).exists()},
}

slug = re.sub(r'[^a-z0-9]+', '_', best['name'].lower()).strip('_')
spark_model_path = MODEL_DIR / f'accident_severity_{slug}_spark'
try:
    fitted_model = fitted[best['name']] if best['name'] in fitted else rf_model
    fitted_model.write().overwrite().save(spark_model_path.as_uri())
    saved_model = str(spark_model_path.relative_to(PROJECT))
except Exception as e:
    print('⚠ модель Spark не сохранена:', type(e).__name__, str(e)[:200])
    saved_model = None

meta_path = MODEL_DIR / f'accident_severity_{slug}_{datetime.now():%Y%m%d_%H%M}.json'
dump_json({'task': TASK, 'model': best['name'], 'alpha': alpha_b, 'features': TREE_NAMES, 'classes': CLASSES,
           'metrics': bm, 'saved_at': datetime.now().isoformat(), 'spark_model': saved_model}, meta_path)

report['artifacts'] = {'model_spark': saved_model, 'model_meta': str(meta_path.relative_to(PROJECT)),
                       'metrics_csv': str((ART_DIR / 'metrics_all_models.csv').relative_to(PROJECT))}
report_path = ART_DIR / 'report.json'
dump_json(report, report_path)


def upsert_index(entry):
    p = PROJECT / 'artifacts' / 'index.json'
    idx = json.loads(p.read_text()) if p.exists() else {'generated_at': None, 'tasks': []}
    idx['tasks'] = sorted([t for t in idx['tasks'] if t['id'] != entry['id']] + [entry], key=lambda t: t['id'])
    idx['generated_at'] = datetime.now().isoformat()
    dump_json(idx, p)


upsert_index({'id': TASK, 'title': report['meta']['title'], 'best_model': best['name'], 'hero_metric': 'f1_macro',
              'hero_value': float(bm['f1_macro']), 'unit': '', 'report': str(report_path.relative_to(PROJECT)),
              'charts_dir': str(ART_DIR.relative_to(PROJECT))})

print('report →', report_path, f'({report_path.stat().st_size / 1024:.0f} КБ)')
print(report['headline']['verdict'])
print('время этапов:', json.dumps(TIMINGS, ensure_ascii=False))

## 9. Итог

* Данные: ~7.7 млн реальных ДТП; чтение 3 ГБ CSV, очистка, признаки, обучение и оценка — на Spark, на драйвер уходят таблицы и выборка в ~1000 точек.
* **Выбор метрики определяется структурой классов.** Accuracy и weighted-F1 почти не отличают baseline «всегда класс 2» от обученных моделей;
  macro-F1, balanced accuracy и per-class recall — отличают.
* Разрыв `weighted-F1 − macro-F1` — быстрый диагноз: чем он больше, тем сильнее усреднение по поддержке прячет провал на редких классах.
* Веса классов и prior-коррекция двигают решение в пользу редких классов — ценой точности на частом. Что важнее — решает задача (цена пропуска тяжёлой аварии).
* Классы порядковые: квадратичный κ показывает, насколько серьёзны ошибки, а не только их число.

**Артефакты:** `artifacts/classification/accident_severity/{report.json, eda.png, macro_vs_weighted.png, confusion_matrix.png, metrics_all_models.csv}`.

Остановить Spark: `spark.stop()`.